# 第四节 基于 Gensim 的词向量实战

## 一、导入库

In [ ]:
# 如果尚未安装，请运行以下命令：
%pip install gensim jieba

In [1]:
import jieba
from gensim import corpora, models
from gensim.models import Word2Vec, KeyedVectors
import warnings

warnings.filterwarnings('ignore')

## 二、Gensim工作流

In [ ]:
# Step 1: 准备分词后的语料 (新闻标题)
raw_headlines = [
    "央行降息，刺激股市反弹",
    "球队赢得总决赛冠军，球员表现出色"
]
# 解释

tokenized_headlines = [jieba.lcut(doc) for doc in raw_headlines]

# Step 2: 创建词典
dictionary = corpora.Dictionary(tokenized_headlines)
print(f"词典: {dictionary}")
# 1. token2id - 核心映射
print("\n【token2id】词到ID的映射")
print("类型:", type(dictionary.token2id))
print("内容:", dictionary.token2id)

# 2. id2token - ID到词的映射（通过属性访问）
print("\n【id2token】ID到词的映射")
print("类型:", type(dictionary.id2token))
print("内容:", dictionary.id2token)

# 3. cfs - 文档频率
print("\n【cfs】文档频率（Document Frequency）")
print("类型:", type(dictionary.cfs))
print("内容:", dictionary.cfs)
print("说明: 键=词ID, 值=该词出现在多少篇文档中")

# 4. dfs - 同 cfs（兼容性）
print("\n【dfs】文档频率（同cfs）")
print("内容:", dictionary.dfs)

# 5. num_docs - 文档总数
print("\n【num_docs】文档总数")
print(dictionary.num_docs)

# 6. num_pos - 总词频
print("\n【num_pos】总词频（所有文档的词数量之和）")
print(dictionary.num_pos)

# 7. num_nnz - 非零词条数（去重后的总词数）
print("\n【num_nnz】非零词条数（词典大小）")
print(dictionary.num_nnz)



In [4]:
# Step 3: 转换为 BoW 向量语料库
corpus_bow = [dictionary.doc2bow(doc) for doc in tokenized_headlines]
# print(f"BoW语料库: {corpus_bow}")
print("=" * 60)
print("BoW 语料库的数据结构")
print("=" * 60)

print(f"\ncorpus_bow 类型: {type(corpus_bow)}")
print(f"corpus_bow 长度: {len(corpus_bow)} (文档数量)")
print(f"\ncorpus_bow 内容: {corpus_bow}")
print("\n" + "=" * 70)
print("逐篇文档解析")
print("=" * 70)

for i, doc_bow in enumerate(corpus_bow):
    print(f"\n【文档{i + 1}的 BoW 表示】")
    print(f"类型: {type(doc_bow)} (列表)")
    print(f"长度: {len(doc_bow)} (包含 {len(doc_bow)} 个不同的词)")
    print(f"内容: {doc_bow}")

    print(f"\n详细解析:")
    for word_id, tf in doc_bow:
        word = dictionary[word_id]
        df = dictionary.cfs[word_id]
        print(f"  ({word_id}, {tf}) → '{word}' 出现 {tf} 次，文档频率={df}")

BoW 语料库的数据结构

corpus_bow 类型: <class 'list'>
corpus_bow 长度: 2 (文档数量)

corpus_bow 内容: [[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1)], [(5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1)]]

逐篇文档解析

【文档1的 BoW 表示】
类型: <class 'list'> (列表)
长度: 6 (包含 6 个不同的词)
内容: [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1)]

详细解析:
  (0, 1) → '刺激' 出现 1 次，文档频率=1
  (1, 1) → '反弹' 出现 1 次，文档频率=1
  (2, 1) → '央行' 出现 1 次，文档频率=1
  (3, 1) → '股市' 出现 1 次，文档频率=1
  (4, 1) → '降息' 出现 1 次，文档频率=1
  (5, 1) → '，' 出现 1 次，文档频率=2

【文档2的 BoW 表示】
类型: <class 'list'> (列表)
长度: 7 (包含 7 个不同的词)
内容: [(5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1)]

详细解析:
  (5, 1) → '，' 出现 1 次，文档频率=2
  (6, 1) → '冠军' 出现 1 次，文档频率=1
  (7, 1) → '总决赛' 出现 1 次，文档频率=1
  (8, 1) → '球员' 出现 1 次，文档频率=1
  (9, 1) → '球队' 出现 1 次，文档频率=1
  (10, 1) → '表现出色' 出现 1 次，文档频率=1
  (11, 1) → '赢得' 出现 1 次，文档频率=1


## 三、TF-IDF与关键词权重

In [5]:
# 1. 准备语料 (新闻标题，包含财经和体育两个明显主题)
headlines = [
    "央行降息，刺激股市反弹",
    "球队赢得总决赛冠军，球员表现出色",
    "国家队公布最新一期足球集训名单",
    "A股市场持续震荡，投资者需谨慎",
    "篮球巨星刷新历史得分记录",
    "理财产品收益率创下新高"
]
tokenized_headlines = [jieba.lcut(title) for title in headlines]

# 2. 创建词典和 BoW 语料库
dictionary = corpora.Dictionary(tokenized_headlines)
corpus_bow = [dictionary.doc2bow(doc) for doc in tokenized_headlines]

print(f"词典大小: {len(dictionary)}")
print(f"语料库包含 {len(corpus_bow)} 篇文档")

词典大小: 35
语料库包含 6 篇文档


In [6]:
# 3. 训练 TF-IDF 模型
tfidf_model = models.TfidfModel(corpus_bow)

# 4. 将 BoW 语料库转换为 TF-IDF 向量表示
corpus_tfidf = tfidf_model[corpus_bow]

In [7]:
# 辅助函数：把 (token_id, weight) 转成 (token, weight)，并按权重降序展示
def tfidf_with_words(tfidf_vec, id2word):
    pairs = [(id2word[token_id], weight) for token_id, weight in tfidf_vec]
    return sorted(pairs, key=lambda x: x[1], reverse=True)


In [8]:
# 打印第一篇标题的 TF-IDF 向量
first_tfidf = list(corpus_tfidf)[0]
print("第一篇标题的TF-IDF向量:")
print(first_tfidf)
print("第一篇标题的TF-IDF向量(带词语):")
print(tfidf_with_words(first_tfidf, dictionary))

第一篇标题的TF-IDF向量:
[(0, np.float64(0.44066740566370055)), (1, np.float64(0.44066740566370055)), (2, np.float64(0.44066740566370055)), (3, np.float64(0.44066740566370055)), (4, np.float64(0.44066740566370055)), (5, np.float64(0.1704734229377651))]
第一篇标题的TF-IDF向量(带词语):
[('刺激', np.float64(0.44066740566370055)), ('反弹', np.float64(0.44066740566370055)), ('央行', np.float64(0.44066740566370055)), ('股市', np.float64(0.44066740566370055)), ('降息', np.float64(0.44066740566370055)), ('，', np.float64(0.1704734229377651))]


In [10]:
# 5. 对新标题应用模型
new_headline = "股市大涨，牛市来了"
new_headline_bow = dictionary.doc2bow(list(jieba.cut(new_headline)))
new_headline_tfidf = tfidf_model[new_headline_bow]
print("\n新标题的TF-IDF向量:")
print(new_headline_tfidf)
print(tfidf_with_words(new_headline_tfidf, dictionary))


新标题的TF-IDF向量:
[(3, np.float64(0.9326446771245245)), (5, np.float64(0.360796211497975))]
[('股市', np.float64(0.9326446771245245)), ('，', np.float64(0.360796211497975))]


## 四、LDA与文档主题挖掘

In [15]:
# 1. 准备语料
headlines = [
    "央行降息，刺激股市反弹",
    "球队赢得总决赛冠军，球员表现出色",
    "国家队公布最新一期足球集训名单",
    "A股市场持续震荡，投资者需谨慎",
    "篮球巨星刷新历史得分记录",
    "理财产品收益率创下新高"
]
tokenized_headlines = [jieba.lcut(title) for title in headlines]
for i, doc in enumerate(tokenized_headlines):
    print(f"文档{i+1}: {doc}")



文档1: ['央行', '降息', '，', '刺激', '股市', '反弹']
文档2: ['球队', '赢得', '总决赛', '冠军', '，', '球员', '表现出色']
文档3: ['国家队', '公布', '最新', '一期', '足球', '集训', '名单']
文档4: ['A股', '市场', '持续', '震荡', '，', '投资者', '需谨慎']
文档5: ['篮球', '巨星', '刷新', '历史', '得分', '记录']
文档6: ['理财产品', '收益率', '创下', '新高']


In [16]:
# 2. 创建词典和 BoW 语料库
dictionary = corpora.Dictionary(tokenized_headlines)
print(f"词典大小: {len(dictionary)}")
print(f"词典示例: {list(dictionary.token2id.items())[:10]}")


词典大小: 35
词典示例: [('刺激', 0), ('反弹', 1), ('央行', 2), ('股市', 3), ('降息', 4), ('，', 5), ('冠军', 6), ('总决赛', 7), ('球员', 8), ('球队', 9)]


In [17]:
corpus_bow = [dictionary.doc2bow(doc) for doc in tokenized_headlines]
print(f"\nBoW 语料库:")
for i, bow in enumerate(corpus_bow):
    print(f"  文档{i+1}: {bow}")



BoW 语料库:
  文档1: [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1)]
  文档2: [(5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1)]
  文档3: [(12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 1)]
  文档4: [(5, 1), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1)]
  文档5: [(25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 1)]
  文档6: [(31, 1), (32, 1), (33, 1), (34, 1)]


In [18]:

# 3. 训练 LDA 模型 (假设需要发现 2 个主题)
lda_model = models.LdaModel(corpus=corpus_bow, id2word=dictionary, num_topics=2, random_state=100)

# 4. 查看模型发现的主题
print("模型发现的 2 个主题及其关键词:")
for topic in lda_model.print_topics():
    print(topic)

模型发现的 2 个主题及其关键词:
(0, '0.045*"，" + 0.040*"公布" + 0.039*"一期" + 0.039*"名单" + 0.039*"足球" + 0.039*"最新" + 0.038*"集训" + 0.038*"国家队" + 0.037*"A股" + 0.037*"市场"')
(1, '0.066*"，" + 0.039*"篮球" + 0.039*"刷新" + 0.039*"历史" + 0.039*"记录" + 0.038*"得分" + 0.038*"巨星" + 0.037*"刺激" + 0.036*"降息" + 0.036*"反弹"')


In [19]:
# 打印所有主题
for topic_idx, topic_words in lda_model.print_topics(num_words=8):
    print(f"\n主题 {topic_idx}:")
    print(f"  {topic_words}")

    # 更详细地展示关键词和权重
    words = topic_words.split(" + ")
    print("  关键词详情:")
    for word_weight in words:
        weight, word = word_weight.split('*')
        word = word.strip('"')
        print(f"    {word}: {float(weight):.4f}")


主题 0:
  0.045*"，" + 0.040*"公布" + 0.039*"一期" + 0.039*"名单" + 0.039*"足球" + 0.039*"最新" + 0.038*"集训" + 0.038*"国家队"
  关键词详情:
    ，: 0.0450
    公布: 0.0400
    一期: 0.0390
    名单: 0.0390
    足球: 0.0390
    最新: 0.0390
    集训: 0.0380
    国家队: 0.0380

主题 1:
  0.066*"，" + 0.039*"篮球" + 0.039*"刷新" + 0.039*"历史" + 0.039*"记录" + 0.038*"得分" + 0.038*"巨星" + 0.037*"刺激"
  关键词详情:
    ，: 0.0660
    篮球: 0.0390
    刷新: 0.0390
    历史: 0.0390
    记录: 0.0390
    得分: 0.0380
    巨星: 0.0380
    刺激: 0.0370


In [21]:
# ==================== 查看文档-主题分布 ====================
print("\n" + "="*60)
print("每篇文档的主题分布")
print("="*60)

for i, bow in enumerate(corpus_bow):
    topic_dist = lda_model[bow]
    print(f"\n文档{i+1}: {headlines[i]}")
    print(f"  主题分布: {topic_dist}")

    # 显示主要主题
    main_topic = max(topic_dist, key=lambda x: x[1])
    print(f"  主要主题: 主题{main_topic[0]} (概率: {main_topic[1]:.4f})")


每篇文档的主题分布

文档1: 央行降息，刺激股市反弹
  主题分布: [(0, np.float32(0.08226446)), (1, np.float32(0.9177356))]
  主要主题: 主题1 (概率: 0.9177)

文档2: 球队赢得总决赛冠军，球员表现出色
  主题分布: [(0, np.float32(0.078730956)), (1, np.float32(0.921269))]
  主要主题: 主题1 (概率: 0.9213)

文档3: 国家队公布最新一期足球集训名单
  主题分布: [(0, np.float32(0.9325611)), (1, np.float32(0.06743891))]
  主要主题: 主题0 (概率: 0.9326)

文档4: A股市场持续震荡，投资者需谨慎
  主题分布: [(0, np.float32(0.920027)), (1, np.float32(0.07997303))]
  主要主题: 主题0 (概率: 0.9200)

文档5: 篮球巨星刷新历史得分记录
  主题分布: [(0, np.float32(0.07686681)), (1, np.float32(0.9231332))]
  主要主题: 主题1 (概率: 0.9231)

文档6: 理财产品收益率创下新高
  主题分布: [(0, np.float32(0.8788)), (1, np.float32(0.121200055))]
  主要主题: 主题0 (概率: 0.8788)


In [23]:
# ==================== 推断新文档 ====================
print("\n" + "="*60)
print("推断新文档的主题分布")
print("="*60)

new_headlines = [
    "巨星詹姆斯获得常规赛MVP",
    "央行宣布再次降息，利好股市",
    "中国男足备战世界杯预选赛"
]

for new_headline in new_headlines:
    print(f"\n新标题: {new_headline}")

    # 分词
    tokenized = jieba.lcut(new_headline)
    print(f"  分词: {tokenized}")

    # 转换为 BoW
    bow = dictionary.doc2bow(tokenized)

    # 推断主题
    topic_dist = lda_model[bow]
    print(f"  主题分布: {topic_dist}")

    # 显示最可能的主题
    if topic_dist:
        main_topic = max(topic_dist, key=lambda x: x[1])
        print(f"  最可能主题: 主题{main_topic[0]} (概率: {main_topic[1]:.4f})")



推断新文档的主题分布

新标题: 巨星詹姆斯获得常规赛MVP
  分词: ['巨星', '詹姆斯', '获得', '常规赛', 'MVP']
  主题分布: [(0, np.float32(0.27250192)), (1, np.float32(0.72749805))]
  最可能主题: 主题1 (概率: 0.7275)

新标题: 央行宣布再次降息，利好股市
  分词: ['央行', '宣布', '再次', '降息', '，', '利好', '股市']
  主题分布: [(0, np.float32(0.117453724)), (1, np.float32(0.88254625))]
  最可能主题: 主题1 (概率: 0.8825)

新标题: 中国男足备战世界杯预选赛
  分词: ['中国男足', '备战', '世界杯', '预选赛']
  主题分布: [(0, np.float32(0.5)), (1, np.float32(0.5))]
  最可能主题: 主题0 (概率: 0.5000)


In [24]:
# ==================== 模型评估 ====================
print("\n" + "="*60)
print("模型评估")
print("="*60)

# 计算困惑度（perplexity，越小越好）
perplexity = lda_model.log_perplexity(corpus_bow)
print(f"困惑度: {perplexity:.4f}")

# 计算主题一致性（coherence，越高越好）
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    model=lda_model,
    texts=tokenized_headlines,
    dictionary=dictionary,
    coherence='c_v'
)
coherence = coherence_model.get_coherence()
print(f"主题一致性: {coherence:.4f}")


模型评估
困惑度: -4.2309
主题一致性: 0.3974


In [ ]:
# ==================== 可视化主题（可选） ====================
try:
    import pyLDAvis
    import pyLDAvis.gensim_models as gensimvis

    print("\n" + "="*60)
    print("主题可视化")
    print("="*60)

    # 准备可视化数据
    vis_data = gensimvis.prepare(lda_model, corpus_bow, dictionary)

    # 保存为 HTML 文件
    pyLDAvis.save_html(vis_data, 'lda_visualization.html')
    print("主题可视化已保存为 'lda_visualization.html'")
    print("请在浏览器中打开该文件查看交互式可视化")

except ImportError:
    print("未安装 pyLDAvis，跳过可视化")
    print("安装命令: pip install pyLDAvis")


In [27]:
# ==================== 保存模型 ====================
print("\n" + "="*60)
print("保存模型")
print("="*60)

# 保存模型
lda_model.save('lda_model.model')
dictionary.save('dictionary.dict')

print("模型已保存:")
print("  - lda_model.model (LDA模型)")
print("  - dictionary.dict (词典)")

# 加载模型示例
# loaded_model = models.LdaModel.load('lda_model.model')
# loaded_dict = corpora.Dictionary.load('dictionary.dict')
# print("\n模型加载成功！")

print("\n" + "="*60)
print("LDA 主题模型分析完成！")
print("="*60)


保存模型
模型已保存:
  - lda_model.model (LDA模型)
  - dictionary.dict (词典)

LDA 主题模型分析完成！


## 五、Word2Vec模型实战

### 5.1 模型训练与核心参数

In [ ]:
# 1. 准备语料 
headlines = [
    # 财经
    "央行降息，刺激股市反弹",
    "A股市场持续震荡，投资者需谨慎",
    "理财产品收益率创下新高",
    "证监会发布新规，规范市场交易",
    "创业板指数上涨，科技股领涨大盘",
    "房价调控政策出台，房地产市场降温",
    "全球股市动荡，影响资本市场信心",
    "分析师认为，当前股市风险与机遇并存，市场情绪复杂",

    # 体育
    "球队赢得总决赛冠军，球员表现出色",
    "国家队公布最新一期足球集训名单",
    "篮球巨星刷新历史得分记录",
    "奥运会开幕，中国代表团旗手确定",
    "马拉松比赛圆满结束，选手创造佳绩",
    "电子竞技联赛吸引大量年轻观众",
    "这支球队的每位球员都表现出色",
    "球员转会市场活跃，多支球队积极引援"
]

tokenized_headlines = [list(jieba.cut(title)) for title in headlines]

# 2. 训练 Word2Vec 模型
model = Word2Vec(tokenized_headlines, vector_size=50, window=3, min_count=1, sg=1)

print(f"模型训练完成！")
print(f"词汇表大小: {len(model.wv)}")
print(f"词向量维度: {model.wv.vector_size}")

# 训练完成后，所有词向量都存储在 model.wv 对象中
# model.wv 是一个 KeyedVectors 实例

### 5.2 使用词向量


In [ ]:
# 1. 寻找最相似的词
# 在小语料上，结果可能不完美，但能体现出模型学习到了主题内的关联
similar_to_market = model.wv.most_similar('股市')
print(f"与 '股市' 最相似的词: {similar_to_market}")

# 2. 计算两个词的余弦相似度
similarity = model.wv.similarity('球队', '球员')
print(f"\n'球队' 和 '球员' 的相似度: {similarity:.4f}")

# 3. 获取一个词的向量
market_vector = model.wv['市场']
print(f"\n'市场' 的向量维度: {market_vector.shape}")

### 5.3 模型的持久化


In [ ]:
# 保存词向量到文件
model.wv.save("news_vectors.kv")

# 从文件加载词向量
loaded_wv = KeyedVectors.load("news_vectors.kv")

# 加载后可以执行同样的操作
print(f"\n加载后，'球队' 和 '球员' 的相似度: {loaded_wv.similarity('球队', '球员'):.4f}")